# Data Access

In [ ]:
"""
Get A specific day of global data from the dataset, from the S3 bucket.
"""
from datetime import date, timedelta
from wekeo_combined_chain.s3_access import get_combined_ds_range

start = date(2025, 8, 2)
end = date(2025, 8, 15)
dates_lst = [start + timedelta(days=i) for i in range((end - start).days + 1)]
ds = get_combined_ds_range(start, end)

In [ ]:
"""
Select a specific area of interest in the dataset.
"""
from wekeo_combined_chain.utils import select_area

areas = {
    "Global":          [ 90., -90.,  180., -180.],
    "North_America":   [ 90.,   9.,  -20., -169.],
    "South_America":   [  9., -60.,  -20.,  120.],
    "Europe":          [ 90.,  36.,   31.,  -20.],
    "Africa":          [ 36., -60.,   60.,  -20.],
    "Russia":          [ 90.,  36., -169.,   31.],
    "Asia":            [ 36., -10., -169.,   60.],
    "Australia":       [-10., -60., -120.,   60.],
    # "Central_Pacific": [  9., -10., -120., -169.],
    # "Antarctic":       [-60., -90.,  180., -180.],
}

area_name = "Global"
area = areas[area_name]

## User defined area override example:
# area_name = "France"
# area = [53., 41., 9., -5.]

ds = select_area(ds, area)

# Post-processing

## - Data list

In [ ]:

from datetime import timedelta
import pandas as pd
from wekeo_combined_chain import postprocess
import os
import sys

results = []  # liste de (date, df_plumes)

dates_present = {pd.Timestamp(t).date(): t for t in ds.time.values}

for d in dates_lst:
    day = d if not hasattr(d, 'date') else d.date()  # à vérifier : type exact de dates_lst

    if day not in dates_present:
        print(f"[NO DATA] {day}: No data found in bucket")
        continue

    t = dates_present[day]
    ds_day = ds.sel(time=t).squeeze()

    try:
        with open(os.devnull, 'w') as devnull:
            old_stdout = sys.stdout
            sys.stdout = devnull
            df_plumes_only = postprocess.compute_plume_stats(ds_day)
            if df_plumes_only.empty or "label" not in df_plumes_only.columns:
                sys.stdout = old_stdout
                print(f"[NO PLUME] {day}: No detected plume")
                continue
            ds_post, df_plumes = postprocess.build_grids(ds_day, df_plumes_only), df_plumes_only
            sys.stdout = old_stdout
        results.append((day, df_plumes))
        print(f"[OK] {day}")
    except Exception as e:
        sys.stdout = old_stdout
        print(f"[ERROR] {day}: {type(e).__name__} - {e}")
        
#------------------------------------------------------------------------

## - Post-process Summary

In [ ]:
# Agréger : une ligne par jour
summary = []
for day, df in results:
    #keep only plumes (not tiny) --> to comment or uncomment:
    #df_p = df[df["label"] < 100]

    # compute % of plumes confirmed / with FRP data:
    sum_pl = 0
    for ipl in range(len(df)):
        if (df['n_frp_cells_total_plume'][ipl] > 0):
            sum_pl+=1
    percent_pl_frp = round(sum_pl/len(df)*100)
    
    summary.append({
        "date"              : day,
        "n_plumes"          : len(df),
        "% of confirmed plumes (with FRP)" : percent_pl_frp,
        "mean_fire_score"   : df["fire_score_MWIR_plume"].mean(), # mean of the fire_score (per plume) over the N plumes detected for a given day
        "max_fire_score"    : df["fire_score_MWIR_plume"].max(), # idem for max
        "n_sources_detected": df["source_is_localized_MWIR_sum_plume"].sum(),
    })

df_summary = pd.DataFrame(summary).set_index("date")
print('--------------------------------')
print('--> Summary for each day:')
print(df_summary)
print('--------------------------------')

# PLOTS

## animation 1: daily S5P mean CO score

In [ ]:
# from wekeo_combined_chain.animation.plot import animate_cartes_journalieres

# animate_cartes_journalieres(ds,
#     varname='s5p_pca__mean_score_CO',
#     date_debut=str(start),
#     date_fin=str(end),
#     duration_ms=400)

## Plot timeseries 

### Timeserie 1: Daily average fire score

In [ ]:
from wekeo_combined_chain.timeseries.plot_bis import plot_fire_score_timeseries

plot_fire_score_timeseries(df_summary)


### Timeserie 2: daily number of sources detected

In [ ]:
from wekeo_combined_chain.timeseries.plot_bis import plot_sources_timeseries
plot_sources_timeseries(df_summary)

### Timeserie 3: plume detection count & pixel counts for plumes

In [ ]:
from wekeo_combined_chain.timeseries.plot_bis import plot_plume_pixels_timeseries, plot_plume_combined_timeseries,plot_plume_total_combined_timeseries

# Plot timeseries of pixel counts for normal plumes
#fig, ax = plot_plume_pixels_timeseries(ds, plume_size="normal")

# Plot timeseries of pixel counts for tiny plumes
#fig, ax = plot_plume_pixels_timeseries(ds, plume_size="tiny")

plot_plume_combined_timeseries(ds)


In [ ]:
from wekeo_combined_chain.timeseries.plot_bis import plot_plume_combined_timeseries, plot_plume_total_combined_timeseries

plot_plume_total_combined_timeseries(ds,df_summary)

In [ ]:
from wekeo_combined_chain.timeseries.plot_bis import plot_plume_cell_coverage_timeseries
plot_plume_cell_coverage_timeseries(ds)

### Timeserie 4: number of daily detected pixels

In [ ]:
from wekeo_combined_chain.timeseries.plot import plot_detected_pixels_timeseries

# Plot timeseries of daily detected pixels (s5p_pca__mean_score_CO not NaN)
fig, ax = plot_detected_pixels_timeseries(ds)

### Timeserie 5: daily FRP pixels

In [ ]:
from wekeo_combined_chain.timeseries.plot import plot_frp_pixels_timeseries

# Plot timeseries of daily FRP pixels (day/night MWIR not NaN)
fig, ax = plot_frp_pixels_timeseries(ds)

## Occurrence Maps

In [ ]:
"""
Coarsen dataset to reduce spatial resolution by block-averaging.
"""

FACTOR = 4 # 12-16 good on a global scale, for regionnal smaller factor advised, around 4-8
dsc = ds.coarsen(latitude=FACTOR, longitude=FACTOR, boundary="trim").mean()

### Occurence Map 1: plume occurence

In [ ]:
from wekeo_combined_chain.timeseries.plot_maps import plot_plume_occurrence_map

fig, ax = plot_plume_occurrence_map(dsc)


### Occurence Map 2: Detections

In [ ]:
from wekeo_combined_chain.timeseries.plot_maps import plot_detection_occurrence_map

fig, ax = plot_detection_occurrence_map(dsc)


### Occurence map 3: Active Fire Occurence

In [ ]:
from wekeo_combined_chain.timeseries.plot_maps import plot_fire_occurrence_map

# Note: uses frp_slstr__day_FRP_MWIR_mean (and/or night) to detect active fire days
fig, ax = plot_fire_occurrence_map(dsc)


## Animated Daily Maps

Browse day-by-day through the spatial maps using an interactive slider.
Each function also accepts `save_gif_path` to export the animation as a GIF file.


In [ ]:
from wekeo_combined_chain.timeseries.plot_maps import animate_plume_map

# Interactive slider — scrub through days
animate_plume_map(dsc)


In [ ]:
# Export as GIF (uncomment to run)
# animate_plume_map(ds, save_gif_path="plume_animation.gif", fps=2, dpi=100)


In [ ]:
from wekeo_combined_chain.timeseries.plot_maps import animate_detection_map

animate_detection_map(dsc)
